<a href="https://colab.research.google.com/github/kachidiniru/Customer_Purchase_data/blob/main/Project_Healthcare_Data_Integration_and_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Healthcare Data Integration and Analysis

This template provides a starting point to help you complete the Tasks 3 and 4 of the project. The questions in this template are numbered in alignment with the Project Overview reading and submission template.
You can use it as-is, modify it, or create your own structure; just be sure to complete all the required tasks.

In [2]:
# Install required packages
import sqlite3
import requests
import pandas as pd
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [1]:
import re             # For regular expression pattern matching in text processing

from datetime import datetime, timedelta  # For parsing and manipulating date/time values
import random  # For generating random values during data exploration

from scipy import stats  # For statistical functions like z-score for outlier detection
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
# StandardScaler: standardizes features (mean=0, std=1) for ML algorithms
# MinMaxScaler: scales features to a fixed range (typically 0-1)
# OneHotEncoder: converts categorical features into binary indicator variables

print("All libraries imported successfully!")
print("Ready to begin healthcare data preprocessing.")

All libraries imported successfully!
Ready to begin healthcare data preprocessing.


# Task 3: Identify and fix the issues in a data set using Python

In [3]:
#load dataset
df = pd.read_csv("https://foundations-of-healthcare-data-analytics-4e579d.gitlab.io/labs/Projects/Patients_EHR_raw_data.csv")

In [4]:
#print top 5 rows
df.head()

,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
0,NV001,1983-03-11,Female,30933.0,Type 2 Diabetes (E11),07-06-2024
1,NV004,NaN,Male,76999.0,Asthma (J45.909),27-10-2022
2,NV005,1984-08-25,Female,40307.0,Osteoarthritis (M15.9),30-10-2024
3,NV006,1982-12-16,Female,56844.0,Type 2 Diabetes (E11),05-07-2022
4,NV007,1990-10-11,Male,NaN,Hyperlipidemia (E78.5),17-06-2024


In [12]:
# Basic dataset structure
print("Dataset dimensions:")
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")
print("\nColumn names and data types:")
print(df.dtypes)

Dataset dimensions:
Number of rows: 74
Number of columns: 6

Column names and data types:
PatientID            object
DateOfBirth          object
Gender               object
ZipCode             float64
PrimaryCondition     object
AdmissionDate        object
dtype: object


### 3.1a. Write the Python code to identify the Number of missing values.

In [6]:
# Check for missing values
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nPercentage of missing values:")
print((df.isnull().sum() / len(df) * 100).round(2))


Missing values per column:
PatientID           0
DateOfBirth         3
Gender              0
ZipCode             4
PrimaryCondition    0
AdmissionDate       0
dtype: int64

Percentage of missing values:
PatientID           0.00
DateOfBirth         4.05
Gender              0.00
ZipCode             5.41
PrimaryCondition    0.00
AdmissionDate       0.00
dtype: float64


In [11]:
missing_values = df.isna().sum()
missing_summary = pd.DataFrame({
    "Missing values": missing_values,
    "Percentage (%)": (missing_values / len(df) * 100).round(2)
})

print(f"Total missing values: {missing_values.sum()}")
display(missing_summary)


Total missing values: 7


,Missing values,Percentage (%)
PatientID,0,0.00
DateOfBirth,3,4.05
Gender,0,0.00
ZipCode,4,5.41
PrimaryCondition,0,0.00
AdmissionDate,0,0.00


There are 7 missing values in total

### 3.1b. Write the Python code to identify the number of duplicate rows.

In [14]:
duplicate_rows_to_remove = df.duplicated().sum()
all_rows_in_duplicate_groups = df.duplicated(keep=False).sum()

duplicate_records = df[df.duplicated(keep=False)].copy()

print(f"Duplicate rows to remove: {duplicate_rows_to_remove}")
print(f"Rows belonging to duplicate groups: {all_rows_in_duplicate_groups}")
display(duplicate_records)


Duplicate rows to remove: 2
Rows belonging to duplicate groups: 4


,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
20,NV028,18-05-1978,Male,NaN,Type 2 Diabetes (E11),27-10-2022
21,NV028,18-05-1978,Male,NaN,Type 2 Diabetes (E11),27-10-2022
43,NV053,1941-04-03,Male,39856.0,Hyperlipidemia (E78.5),17-06-2024
44,NV053,1941-04-03,Male,39856.0,Hyperlipidemia (E78.5),17-06-2024


### 3.1c Write the Python code to identify inconsistent categorical entries in the Gender column.

In [15]:
print("Raw Gender values and counts:")
display(df["Gender"].value_counts(dropna=False).rename_axis("Gender").reset_index(name="Count"))

valid_standard_values = {"Female", "Male"}
inconsistent_gender = df.loc[
    ~df["Gender"].isin(valid_standard_values),
    ["PatientID", "Gender"]
]

print("Inconsistent Gender entries:")
display(inconsistent_gender)


Raw Gender values and counts:


,Gender,Count
0,Female,34
1,Male,34
2,F,4
3,M,1
4,male,1


Inconsistent Gender entries:


,PatientID,Gender
10,NV015,M
29,NV036,male
53,NV070,F
63,NV085,F
70,NV093,F
71,NV094,F


### 3.1d Write the Python code to identify inconsistent date formats in the AdmissionDate column.

In [16]:
def classify_admission_date(value):
    value = str(value).strip()
    if re.fullmatch(r"\d{2}-\d{2}-\d{4}", value):
        return "DD-MM-YYYY"
    return "Invalid / inconsistent"

format_summary = (
    df["AdmissionDate"]
    .apply(classify_admission_date)
    .value_counts()
    .rename_axis("Format")
    .reset_index(name="Count")
)

display(format_summary)

invalid_admission_dates = df.loc[
    df["AdmissionDate"].apply(classify_admission_date) == "Invalid / inconsistent",
    ["PatientID", "AdmissionDate"]
]

print("Invalid/inconsistent AdmissionDate values:")
display(invalid_admission_dates)


,Format,Count
0,DD-MM-YYYY,72
1,Invalid / inconsistent,2


Invalid/inconsistent AdmissionDate values:


,PatientID,AdmissionDate
5,NV008,45258
37,NV046,45439.00


### 3.2a Write the Python code to handle missing values appropriately.

In [18]:
# Create a separate cleaned dataset so df remains unchanged.
df_clean = df.copy()

# Missing dates are retained as missing (NaT after conversion). Do not invent dates.
# ZipCode is an identifier, so use an explicit value rather than a numerical mean/median.
df_clean["ZipCode"] = (
    df_clean["ZipCode"]
    .astype("Int64")
    .astype("string")
    .str.zfill(5)
    .fillna("Unknown")
)
print("Missing values after handling ZipCode:")
print(df_clean.isna().sum())



Missing values after handling ZipCode:
PatientID           0
DateOfBirth         3
Gender              0
ZipCode             0
PrimaryCondition    0
AdmissionDate       0
dtype: int64


### 3.2b Write the Python code to remove exact duplicate rows.

In [19]:
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
rows_after = len(df_clean)

print(f"Rows before removing duplicates: {rows_before}")
print(f"Rows after removing duplicates: {rows_after}")
print(f"Exact duplicate rows removed: {rows_before - rows_after}")


Rows before removing duplicates: 74
Rows after removing duplicates: 72
Exact duplicate rows removed: 2


### 3.2c Write the Python code to standardize gender values.

In [20]:
gender_mapping = {
    "female": "Female",
    "f": "Female",
    "male": "Male",
    "m": "Male"
}

df_clean["Gender"] = (
    df_clean["Gender"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(gender_mapping)
    .fillna("Unknown")
)

print("Standardized Gender values:")
display(df_clean["Gender"].value_counts(dropna=False).rename_axis("Gender").reset_index(name="Count"))


Standardized Gender values:


,Gender,Count
0,Female,38
1,Male,34


### 3.2d Write the Python code to standardize date values.

In [21]:
def parse_mixed_admission_date(value):
    if pd.isna(value):
        return pd.NaT

    value = str(value).strip()

    # Convert Excel serial numbers such as 45258 and 45439.00.
    if re.fullmatch(r"\d+(\.0+)?", value):
        return pd.Timestamp("1899-12-30") + pd.to_timedelta(float(value), unit="D")

    # Parse dates such as 07-06-2024 as DD-MM-YYYY.
    return pd.to_datetime(value, format="%d-%m-%Y", errors="coerce")

# DateOfBirth has mixed ISO and DD-MM-YYYY formats.
df_clean["DateOfBirth"] = pd.to_datetime(
    df_clean["DateOfBirth"],
    format="mixed",
    dayfirst=True,
    errors="coerce"
)
df_clean["AdmissionDate"] = df_clean["AdmissionDate"].apply(parse_mixed_admission_date)

# Display dates consistently as YYYY-MM-DD for review/export.
df_clean["DateOfBirth"] = df_clean["DateOfBirth"].dt.strftime("%Y-%m-%d")
df_clean["AdmissionDate"] = df_clean["AdmissionDate"].dt.strftime("%Y-%m-%d")

print("Final missing values:")
print(df_clean.isna().sum())
display(df_clean.head())

df_clean.to_csv("Patients_EHR_cleaned.csv", index=False)
print("Saved: Patients_EHR_cleaned.csv")



Final missing values:
PatientID           0
DateOfBirth         3
Gender              0
ZipCode             0
PrimaryCondition    0
AdmissionDate       0
dtype: int64


,PatientID,DateOfBirth,Gender,ZipCode,PrimaryCondition,AdmissionDate
0,NV001,1983-03-11,Female,30933,Type 2 Diabetes (E11),2024-06-07
1,NV004,NaN,Male,76999,Asthma (J45.909),2022-10-27
2,NV005,1984-08-25,Female,40307,Osteoarthritis (M15.9),2024-10-30
3,NV006,1982-12-16,Female,56844,Type 2 Diabetes (E11),2022-07-05
4,NV007,1990-10-11,Male,Unknown,Hyperlipidemia (E78.5),2024-06-17


Saved: Patients_EHR_cleaned.csv


Final Task 3 check

In [22]:
print(f"Final dataset shape: {df_clean.shape}")
print("\nFinal missing values:")
print(df_clean.isna().sum())
print("\nFinal duplicate-row count:")
print(df_clean.duplicated().sum())
print("\nFinal Gender values:")
print(df_clean["Gender"].value_counts(dropna=False))


Final dataset shape: (72, 6)

Final missing values:
PatientID           0
DateOfBirth         3
Gender              0
ZipCode             0
PrimaryCondition    0
AdmissionDate       0
dtype: int64

Final duplicate-row count:
1

Final Gender values:
Gender
Female    38
Male      34
Name: count, dtype: int64


# Task 4: Analyze the data sets using SQL

## Medical Data SQLite Exploration
Before you write the queries listed in Task 4 of the project, execute the cells in this section to inspect table structure and query sample rows.

In [23]:
db_url = "https://foundations-of-healthcare-data-analytics-4e579d.gitlab.io/labs/medical_data.db"  # replace
db_path = Path("medical_data.db")

In [ ]:
response = requests.get(db_url)
response.raise_for_status()
db_path.write_bytes(response.content)
print("Downloaded to", db_path)

Downloaded to medical_data.db


In [ ]:
conn = sqlite3.connect(db_path)
cur = conn.cursor()

In [25]:
def run_query(query):
    result = pd.read_sql_query(query, conn)
    display(result)
    print(f"Rows returned: {len(result)}")
    return result

print("Database downloaded and connected successfully.")


Database downloaded and connected successfully.


### Table structure: pharmacy_claims

In [ ]:
run_query("PRAGMA table_info(pharmacy_claims);")

,cid,name,type,notnull,dflt_value,pk
0,0,ClaimID,TEXT,0,None,0
1,1,PatientID,TEXT,0,None,0
2,2,Medication,TEXT,0,None,0
3,3,Dosage,TEXT,0,None,0
4,4,FillDate,TIMESTAMP,0,None,0
5,5,Payer,TEXT,0,None,0
6,6,ClinicID,TEXT,0,None,0
7,7,ChargeAmount,REAL,0,None,0
8,8,PaidAmount,REAL,0,None,0


### Table structure: lab_results

In [ ]:
run_query("PRAGMA table_info(lab_results);")

,cid,name,type,notnull,dflt_value,pk
0,0,PatientID,TEXT,0,None,0
1,1,LabTestID,TEXT,0,None,0
2,2,CollectionDate,TIMESTAMP,0,None,0
3,3,TestName,TEXT,0,None,0
4,4,TestResultValue,REAL,0,None,0
5,5,Units,TEXT,0,None,0
6,6,ReferenceRangeLow,REAL,0,None,0
7,7,ReferenceRangeHigh,REAL,0,None,0
8,8,AbnormalFlag,TEXT,0,None,0


### Table structure: patients

In [ ]:
run_query("PRAGMA table_info(patients);")

,cid,name,type,notnull,dflt_value,pk
0,0,PatientID,TEXT,0,None,0
1,1,DateOfBirth,TIMESTAMP,0,None,0
2,2,Gender,TEXT,0,None,0
3,3,ZipCode,INTEGER,0,None,0
4,4,PrimaryCondition,TEXT,0,None,0
5,5,AdmissionDate,TIMESTAMP,0,None,0


### Top 5 patients

## Write the SQL queries and execute them.
The questions are numbered in alignment with the Project Overview reading and submission template.

### 4.1a Write an SQL query to list all patients born before 1980.  

In [29]:
query_4_1a = """
SELECT PatientID, DateOfBirth, Gender, ZipCode, PrimaryCondition, AdmissionDate
FROM patients
WHERE date(DateOfBirth) < date('1980-01-01')
ORDER BY date(DateOfBirth), PatientID;
"""

### 4.1b Write an SQL query to list all female patients with Type 2 Diabetes.

In [30]:
query_4_1b = """
SELECT PatientID, DateOfBirth, Gender, ZipCode, PrimaryCondition, AdmissionDate
FROM patients
WHERE lower(trim(Gender)) = 'female'
  AND PrimaryCondition LIKE '%Type 2 Diabetes%'
ORDER BY PatientID;
"""


### 4.1c Write an SQL query to display all lab tests performed. List each patient (PatientID, Gender, and DateOfBirth from *patients*) with their Lab test name and result value (TestName and TestResultValue from *lab_results*).

In [32]:
query_4_1c = """
SELECT
    p.PatientID,
    p.Gender,
    p.DateOfBirth,
    l.TestName,
    l.TestResultValue
FROM patients AS p
INNER JOIN lab_results AS l
    ON p.PatientID = l.PatientID
ORDER BY p.PatientID, l.CollectionDate, l.TestName;
"""




### 4.1d Write an SQL query to display all distinct patients with their payer(s). List each patient (PatientID, Gender, and DateofBirth from *patients*) with their payer(s) from *pharmacy_claims*).

In [33]:
query_4_1d = """
SELECT DISTINCT
    p.PatientID,
    p.Gender,
    p.DateOfBirth,
    pc.Payer
FROM patients AS p
INNER JOIN pharmacy_claims AS pc
    ON p.PatientID = pc.PatientID
ORDER BY p.PatientID, pc.Payer;
"""
